# 02 - Preprocessing (cleaning, integration, selection, transformation)

In [1]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 90)
def show(f, w=900): display(Image(filename=str(ROOT / 'outputs' / 'plots' / f), width=w))

In [2]:
from src.preprocessing import pipeline
stats = pipeline.run()
stats

2026-11-21 12:59:32,559 | preprocessing | INFO | PHASE 4 - preprocessing pipeline started


2026-11-21 13:00:02,665 | preprocessing | INFO | cleaned: movies=11410 casts=45049 actors=6791 people=3299 remakes=1249


2026-11-21 13:00:30,762 | preprocessing | INFO | PHASE 4 complete: {'raw_main_rows': 11477, 'raw_main_malformed': 871, 'films_final': 11408, 'directors_profiled': 682, 'casts': 45049, 'actors': 6791, 'people': 3299, 'remakes': 1249, 'transactions': 11408, 'items': 65, 'avg_tx_len': 5.117198457223002, 'dt_rows': 2801, 'dt_features': 58, 'dt_positive_rate': 0.32452695465905035, 'kmeans_features': 19, 'movie_matrix_shape': [11408, 51]}


{'raw_main_rows': 11477,
 'raw_main_malformed': 871,
 'films_final': 11408,
 'directors_profiled': 682,
 'casts': 45049,
 'actors': 6791,
 'people': 3299,
 'remakes': 1249,
 'transactions': 11408,
 'items': 65,
 'avg_tx_len': 5.117198457223002,
 'dt_rows': 2801,
 'dt_features': 58,
 'dt_positive_rate': 0.32452695465905035,
 'kmeans_features': 19,
 'movie_matrix_shape': [11408, 51]}

## Cleaning log - every operation and the records it affected

In [3]:
pd.read_csv('outputs/cleaning_log.csv')

,table,step,attribute,description,records_affected,action
0,MAIN,load,*,Data rows parsed from main.html (<tr><td> lines),11477,loaded
1,MAIN,noise,*,"Rows with <4 fields, empty film_id or no title cell (corrupted rows)",26,removed
2,MAIN,noise,all fields,Rows with != 11 cells (merged/split HTML cells) realigned by monotone slot matching on...,849,realigned
3,MAIN,missing,title,Empty or 'unknown' title (record cannot be identified),12,removed
4,MAIN,missing,year,"Year missing or unparseable (e.g. '19xx', blank)",22,imputed later (group-based)
5,MAIN,invalid,year,Year outside plausible range 1880-1999 (dataset frozen 1999),0,set to missing
6,MAIN,missing,director,"Pseudo-director placeholders (Unknown*, UnYear) - director unknown",218,flagged (director_known=0)
7,MAIN,missing,producers,"Producer unknown (PU:, empty PN:)",6569,constant imputation: 'unknown' category
8,MAIN,inconsistent,studios,"Inconsistent comma spacing in studio names (e.g. 'Shamley,CBS' vs 'Shamley, CBS') stan...",265,standardised
9,MAIN,missing,studios,"Studio unknown (SU:, blank)",5941,kept as missing; 'Unknown' category in categorical encodings


## Outliers (flagged, not deleted) and smoothing

In [4]:
pd.read_csv('outputs/outlier_summary.csv')

,attribute,n,q1,q3,iqr,lower_fence,upper_fence,n_iqr_outliers,n_z_outliers,min,max,verdict
0,year,11408,1946.0,1984.0,38.0,1889.00,2041.00,0,9,1891.0,1999.0,"Genuine: early-cinema films 1891-1910 (Edison, Lumiere, Melies) are real historic reco..."
1,cast_size,8419,3.0,7.0,4.0,-3.00,13.00,207,105,1.0,46.0,"Genuine: large documented casts (epics, Hitchcock films) - kept; reflects documentatio..."
2,director_n_films,11191,3.0,16.0,13.0,-16.50,35.50,337,184,1.0,79.0,Genuine: prolific directors (Hitchcock is complete by design of the dataset) - kept.
3,length_min,560,30.0,60.0,30.0,-15.00,105.00,82,14,1.0,570.0,Mostly genuine epics/serials > 3 h; values outside 1-600 min were unit errors and were...
4,n_producers,11408,0.0,1.0,1.0,-1.50,2.50,94,94,0.0,4.0,Genuine: co-productions with many producers - kept.
5,halliwell_stars,2801,0.0,2.0,2.0,-3.00,5.00,0,0,0.0,4.0,No outliers possible (bounded 0-4).
6,female_cast_share,7224,0.0,0.5,0.5,-0.75,1.25,0,0,0.0,1.0,"Bounded [0, 1]; 0 and 1 are genuine (all-male / all-female documented casts)."


In [5]:
sm = pd.read_csv('data/processed/length_min_binning_smoothing.csv', index_col=0)
sm.groupby('bin').agg(n=('original','size'), lo=('original','min'), hi=('original','max'), mean=('by_mean','first'))

,n,lo,hi,mean
bin,,,,
1,70,1.0,30.0,17.657143
2,70,30.0,30.0,30.000000
3,70,30.0,30.0,30.000000
4,70,30.0,30.0,30.000000
5,70,30.0,40.0,30.785714
6,70,40.0,60.0,57.442857
7,70,60.0,145.0,72.828571
8,70,150.0,570.0,227.428571


## Attribute selection

In [6]:
pd.read_csv('outputs/data_selection.csv')

,attribute,decision,used_by,reason
0,film_id,discard (identifier),-,"Surrogate key composed of director code + sequence number; no analytical meaning, used..."
1,title,discard (identifier),descriptive only,"Free text, ~10 000 unique values; kept for display"
2,year,keep,"descriptive, DT, K-Means (via director profile), transactions (as era)","Only temporal attribute; drives colour process, genre trends and studio era"
3,director,discard as feature,"integration key, director-level clustering unit",2 900+ distinct values (high cardinality identifier); used to join PEOPLE and aggregate
4,producer_raw,transform,"DT (producer_status, n_producers)",Raw names are high-cardinality; the coding (referenced / named / none / unknown) and c...
5,studio,transform,"descriptive, Apriori, DT (studio_group)",1 200+ raw names reduced to the 8 classical Hollywood majors + Hitchcock TV + Other/Un...
6,color_type,keep,"descriptive, Apriori, DT, K-Means","Film process (silent b/w, b/w, colour, colourised, unknown) - technological era marker"
7,genres,keep (multi-valued),"descriptive, Apriori, COFI, DT, K-Means","Core content attribute; multi-valued, natural basket items"
8,halliwell_stars,keep,"descriptive, Apriori, DT target",Critic rating 0-4 stars in Halliwell's Film Guide; basis of the classification target
9,oscar_wins / oscar_noms,"keep for description, EXCLUDE from DT","descriptive, Apriori, K-Means",Awards are outcomes of the same critical judgement as the target -> target leakage in DT


## Transformed outputs

In [7]:
pd.read_csv('data/processed/films_transformed.csv').head()

,film_id,era_code,year_z,cast_size_z,n_producers_z,n_genres_z,director_n_films_filled_z,female_cast_share_filled_z,year_mm,cast_size_mm,n_producers_mm,n_genres_mm,director_n_films_filled_mm,female_cast_share_filled_mm,color_type_bw,color_type_color,color_type_colorized,color_type_silent_bw,color_type_unknown,studio_group_Columbia,...,genre_Dram,genre_Comd,genre_Susp,genre_Romt,genre_Horr,genre_Musc,genre_Actn,genre_Advt,genre_BioP,genre_Docu,genre_West,genre_ScFi,genre_Fant,genre_Porn,genre_Myst,genre_CnR,genre_Cart,genre_Noir,genre_Epic,genre_Hist
0,H1,0,-1.833157,-0.497528,0.774042,0.344659,-0.545056,0.698742,0.287037,0.043478,0.25,0.25,0.051282,0.500000,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,H2,0,-1.833157,-0.497528,0.774042,-1.395878,5.974793,0.698742,0.287037,0.043478,0.25,0.00,1.000000,0.500000,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,H3,0,-1.833157,0.559812,0.774042,0.344659,5.974793,0.698742,0.287037,0.130435,0.25,0.25,1.000000,0.500000,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,H4,0,-1.746291,1.352817,0.774042,-1.395878,5.974793,-0.010151,0.305556,0.195652,0.25,0.00,1.000000,0.333333,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,H5,0,-1.702858,-0.233193,0.774042,-1.395878,5.974793,-0.010151,0.314815,0.065217,0.25,0.00,1.000000,0.333333,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [8]:
pd.read_csv('data/transactions/film_transactions.csv').head(10)

,film_id,items
0,H1,cast 1-2;color=silent_bw;dir_country=UK;era=Silent (<=1927);genre=Drama
1,H2,cast 1-2;color=silent_bw;dir_country=UK;era=Silent (<=1927)
2,H3,cast 6-9;color=silent_bw;dir_country=UK;era=Silent (<=1927);genre=Drama;later_remade=yes
3,H4,cast 6-9;color=silent_bw;dir_country=UK;era=Silent (<=1927)
4,H5,cast 3-5;color=silent_bw;dir_country=UK;era=Silent (<=1927)
5,H7,cast 6-9;color=silent_bw;dir_country=UK;era=Silent (<=1927);genre=Drama;halliwell=cite...
6,H8,cast 3-5;color=silent_bw;dir_country=UK;era=Silent (<=1927);genre=Drama;halliwell=cite...
7,H9,cast 6-9;color=silent_bw;dir_country=UK;era=Silent (<=1927);genre=Suspense/thriller;la...
8,H10,cast 10+;color=silent_bw;dir_country=UK;era=Silent (<=1927);genre=Suspense/thriller
9,H11,cast 10+;color=silent_bw;dir_country=UK;era=Silent (<=1927);genre=Suspense/thriller;so...


In [9]:
print(open('outputs/preprocessing_report.md', encoding='utf-8').read()[:4000])

# Preprocessing Report

## 1. Original dataset size
* MAIN: 11477 data rows (871 malformed, i.e. not 11 cells)
* CASTS: 46197 data rows
* ACTORS: 6818 data rows
* PEOPLE: 3311 data rows
* REMAKES: 1277 data rows
* STUDIOS: 207 data rows

## 2-5. Cleaning operations (missing values, duplicates, outliers, removals)

| table       | step         | attribute           | description                                                                                                                       |   records_affected | action                                                       |
|:------------|:-------------|:--------------------|:----------------------------------------------------------------------------------------------------------------------------------|-------------------:|:-------------------------------------------------------------|
| MAIN        | load         | *                   | Data rows parsed from main.html (<tr><td> lines)                                             